In [ ]:
from vislearnlabpy.embeddings.generate_embeddings import EmbeddingGenerator
from vislearnlabpy.embeddings.embedding_store import EmbeddingStore
from vislearnlabpy.embeddings.similarity_utils import csv_to_text_pairs, plot_rdm
import os

In [2]:
# running this on an M2 mac 
generator = EmbeddingGenerator(device="mps", model_type="clip", output_type= "doc")
PROJECT_PATH = os.getcwd()
input_csv = os.path.join(PROJECT_PATH, "data", "peekbank_stimuli.csv")
# path to the images
input_path = os.path.join(PROJECT_PATH, "data")

In [ ]:
generator.generate_image_embeddings(output_path="embeddings", input_csv=input_csv, input_dir=input_path, batch_size=100, id_column="unique_pair")

In [ ]:
image_embedding_store = EmbeddingStore.from_doc("embeddings/image_embeddings/clip_image_embeddings_doc.docs")
text_embedding_store = EmbeddingStore.from_doc("embeddings/text_embeddings/clip_text_embeddings_doc.docs")

In [ ]:
from vislearnlabpy.embeddings.utils import display_search_results
docs, scores = image_embedding_store.search_store(text_query="dog")
display_search_results(docs, scores)

In [4]:
import pandas as pd
def csv_to_path_pairs(path_pair_csv):
    df = pd.read_csv(path_pair_csv)
    cwd = input_path
    return [(os.path.join(cwd, img1), os.path.join(cwd, img2)) for img1, img2 in df[['image1', 'image2']].itertuples(index=False, name=None)]

In [ ]:
image_df = image_embedding_store.retrieve_similarities(text_pairs=csv_to_path_pairs(input_csv))
text_df = text_embedding_store.retrieve_similarities(text_pairs=csv_to_text_pairs(input_csv))

Missing pairs being skipped above are from Frank 2015, which has empty descriptions for novel words.

Merging image and text similarity files.

In [9]:
input_csv_df = pd.read_csv(input_csv)
input_csv_df = input_csv_df.drop_duplicates(subset=['image1', 'image2'])
input_csv_df["image1"] = input_path + "/" +input_csv_df["image1"]
input_csv_df["image2"] = input_path + "/" + input_csv_df["image2"]
image_df = image_df.rename(columns={'text1': 'image1', 'text2': 'image2'})
image_df = image_df.drop_duplicates(subset=['image1', 'image2'])
merged_df = pd.merge(
    input_csv_df,
    image_df,
    on=['image1', 'image2'],
    how='left'  # or 'left', 'right', 'outer' depending on your need
)
text_df_clean = text_df.drop_duplicates(subset=['text1', 'text2'])
merged_df = merged_df.rename(columns={'cosine_similarity': 'image_similarity'})
full_df = pd.merge(
    merged_df,
    text_df_clean,
    on=['text1', 'text2'],
    how='left'
)
full_df = full_df.rename(columns={'cosine_similarity': 'text_similarity'})
full_df.to_csv("data/similarities-clip_data.csv", index=False)

In [6]:
import numpy as np
plot_rdm("rdm", np.stack(image_embedding_store.EmbeddingList.embedding))